In [340]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [341]:
import sys

sys.path.append("..")

In [342]:

from constrerl.utils import extract_flags_from_name
import glob
from pathlib import Path
import json
import pandas as pd
from collections.abc import Callable, Awaitable


In [354]:
test_file = Path("./results_test/all.csv")
test_df = pd.read_csv(test_file)
report_dir = Path("report")

# test_df = test_df[test_df["Team ID"] == "TUGW"]
# report_dir = report_dir / "test-tugw"
# lbl_xtra = ":test:tugw"
# mode = "Merged Test"
test_df = test_df[test_df["Team ID"] == "ToGS"]
report_dir = report_dir / "test"
lbl_xtra = ":test:togs"
mode = "Test"

report_dir.mkdir(exist_ok=True, parents=True)
test_df

,Team ID,Task ID,Run ID,System Description,macro_precision,macro_recall,macro_f1,micro_precision,micro_recall,micro_f1
69,ToGS,T611,hermesbeamnonehermes318Bbase,CHASTE,0.103839,0.029667,0.023255,0.034678,0.018162,0.023838
70,ToGS,T611,hermesbeamnonehermes323Bbase,CHASTE,0.045921,0.019983,0.012508,0.030994,0.012602,0.017918
71,ToGS,T611,hermesloraentitiesnaivebeamnonehermes323Bentities,CHASTE,0.323552,0.391645,0.333386,0.342805,0.554485,0.423676
72,ToGS,T611,hermesnaivebeamnonehermes318Bbase,CHASTE,0.276523,0.415946,0.307339,0.289350,0.480356,0.361154
73,ToGS,T611,hermesnaivebeamnonehermes323Bbase,CHASTE,0.280636,0.417106,0.314909,0.300520,0.492216,0.373191
...,...,...,...,...,...,...,...,...,...,...
409,ToGS,T622,hermesraglorasbeamnonehermes323Bs,CHASTE,0.017780,0.025772,0.010901,0.048892,0.052675,0.050713
410,ToGS,T622,hermesraglorasnaivebeamnonehermes318Bs,CHASTE,0.020579,0.032821,0.014476,0.054217,0.059259,0.056626
411,ToGS,T622,hermesraglorasnaivebeamnonehermes323Bs,CHASTE,0.017780,0.025772,0.010901,0.048892,0.052675,0.050713
412,ToGS,T622,hermesragnaivebeamnonehermes318Bbase,CHASTE,0.051441,0.037719,0.034030,0.036324,0.054321,0.043536


In [355]:
eval_results: list[dict] = []


score_map = {
    "macro_precision": "$P$",
    "macro_recall": "$R$",
    "macro_f1": "$F_1$",
    "micro_precision": "$P_{micro}$",
    "micro_recall": "$R_{micro}$",
    "micro_f1": "$F_{1,micro}$",
}


def test_table_to_df(
    table: pd.DataFrame,
    task: str,
) -> pd.DataFrame:
    eval_results = []
    for i, row in table.iterrows():
        run_id: str = row["Run ID"]
        merge_mode = row["Team ID"] == "TUGW"
        if row["Task ID"] != task:
            continue
        eval_result = {f"{k}": v for k, v in row.items() if k in score_map}

        result_dict = extract_flags_from_name(
            run_id, merge_mode=merge_mode, k=None, test_mode=True
        )
        result_dict.update(eval_result)
        eval_results.append(result_dict)
    if len(eval_results) == 0:
        return pd.DataFrame()
    eval_df = pd.DataFrame(eval_results)
    # remove duplicate rows
    eval_df = eval_df.drop_duplicates()
    eval_df.rename(score_map, axis=1, inplace=True)
    valid_cols = [
        c
        for c in [
            "Graphwise",
            "Set",
            "Model",
            "Beams",
            "NE FT",
            "RAG",
            "LoRA",
            "Naive",
            "Filter",
        ]
        if c in eval_df.columns
    ]
    eval_df.set_index(valid_cols, inplace=True)
    eval_df = eval_df.sort_index()
    # if "$F_{1,micro}$" in eval_df.columns:
    #     eval_df = eval_df.sort_values("$F_{1,micro}$")
    return eval_df


task_6_1_1_df = test_table_to_df(test_df, "T611")
task_6_1_2_df = test_table_to_df(test_df, "T612")
task_6_2_1_df = test_table_to_df(test_df, "T621")
task_6_2_2_df = test_table_to_df(test_df, "T622")
task_6_1_1_df

$P$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter                 
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.103839   
                                                 \checkmark $\times$    0.276523   
                           \checkmark $\times$   $\times$   $\times$    0.374668   
                                                 \checkmark $\times$    0.335337   
                                      \checkmark $\times$   $\times$    0.197714   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.306500   
                                      \checkmark \checkmark $\times$    0.305095   
                           \checkmark $\times$   $\times$   $\times$    0.003497   
                                      \checkmark \checkmark $\times$    0.327929   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.045921   
                                                 \checkmark $\times$    0.280636   
                                      \checkmark \checkmark $\times$    0.323552   
                           \checkmark $\times$   $\times$   $\times$    0.322846   
                                                 \checkmark $\times$    0.333952   
                                      \checkmark $\times$   $\times$    0.113513   
                                                 \checkmark $\times$    0.318824   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.300719   
                                      \checkmark \checkmark $\times$    0.332628   
                           \checkmark $\times$   $\times$   $\times$    0.004274   
                                      \checkmark \checkmark $\times$    0.326303   
Naive  $\times$ $\times$   $\times$   $\times$   \checkmark $\times$    0.307182   
                                                            \checkmark  0.284171   

                                                                             $R$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter                 
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.029667   
                                                 \checkmark $\times$    0.415946   
                           \checkmark $\times$   $\times$   $\times$    0.223773   
                                                 \checkmark $\times$    0.450644   
                                      \checkmark $\times$   $\times$    0.070961   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.447209   
                                      \checkmark \checkmark $\times$    0.453021   
                           \checkmark $\times$   $\times$   $\times$    0.001789   
                                      \checkmark \checkmark $\times$    0.448721   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.019983   
                                                 \checkmark $\times$    0.417106   
                                      \checkmark \checkmark $\times$    0.391645   
                           \checkmark $\times$   $\times$   $\times$    0.174418   
                                                 \checkmark $\times$    0.441260   
                                      \checkmark $\times$   $\times$    0.069524   
                                                 \checkmark $\times$    0.389665   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.442909   
                                      \checkmark \checkmark $\times$    0.449454   
                           \checkmark $\time

In [356]:
task_6_1_1_df

$P$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter                 
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.103839   
                                                 \checkmark $\times$    0.276523   
                           \checkmark $\times$   $\times$   $\times$    0.374668   
                                                 \checkmark $\times$    0.335337   
                                      \checkmark $\times$   $\times$    0.197714   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.306500   
                                      \checkmark \checkmark $\times$    0.305095   
                           \checkmark $\times$   $\times$   $\times$    0.003497   
                                      \checkmark \checkmark $\times$    0.327929   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.045921   
                                                 \checkmark $\times$    0.280636   
                                      \checkmark \checkmark $\times$    0.323552   
                           \checkmark $\times$   $\times$   $\times$    0.322846   
                                                 \checkmark $\times$    0.333952   
                                      \checkmark $\times$   $\times$    0.113513   
                                                 \checkmark $\times$    0.318824   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.300719   
                                      \checkmark \checkmark $\times$    0.332628   
                           \checkmark $\times$   $\times$   $\times$    0.004274   
                                      \checkmark \checkmark $\times$    0.326303   
Naive  $\times$ $\times$   $\times$   $\times$   \checkmark $\times$    0.307182   
                                                            \checkmark  0.284171   

                                                                             $R$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter                 
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.029667   
                                                 \checkmark $\times$    0.415946   
                           \checkmark $\times$   $\times$   $\times$    0.223773   
                                                 \checkmark $\times$    0.450644   
                                      \checkmark $\times$   $\times$    0.070961   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.447209   
                                      \checkmark \checkmark $\times$    0.453021   
                           \checkmark $\times$   $\times$   $\times$    0.001789   
                                      \checkmark \checkmark $\times$    0.448721   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.019983   
                                                 \checkmark $\times$    0.417106   
                                      \checkmark \checkmark $\times$    0.391645   
                           \checkmark $\times$   $\times$   $\times$    0.174418   
                                                 \checkmark $\times$    0.441260   
                                      \checkmark $\times$   $\times$    0.069524   
                                                 \checkmark $\times$    0.389665   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.442909   
                                      \checkmark \checkmark $\times$    0.449454   
                           \checkmark $\time

In [357]:
task_6_1_1_df.style.highlight_max(axis=0, props="textbf:--rwrap;").format(precision=2).to_latex(
    report_dir / "task_6_1_1.tex",
    # float_format="%.2f",
    caption=f"{mode} Set Result for Task 6.1.1 for various models and approaches.",
    label=f"tab:task:6_1_1{lbl_xtra}",
    clines="all;data",
    hrules=True
)
task_6_1_1_df

$P$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter                 
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.103839   
                                                 \checkmark $\times$    0.276523   
                           \checkmark $\times$   $\times$   $\times$    0.374668   
                                                 \checkmark $\times$    0.335337   
                                      \checkmark $\times$   $\times$    0.197714   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.306500   
                                      \checkmark \checkmark $\times$    0.305095   
                           \checkmark $\times$   $\times$   $\times$    0.003497   
                                      \checkmark \checkmark $\times$    0.327929   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.045921   
                                                 \checkmark $\times$    0.280636   
                                      \checkmark \checkmark $\times$    0.323552   
                           \checkmark $\times$   $\times$   $\times$    0.322846   
                                                 \checkmark $\times$    0.333952   
                                      \checkmark $\times$   $\times$    0.113513   
                                                 \checkmark $\times$    0.318824   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.300719   
                                      \checkmark \checkmark $\times$    0.332628   
                           \checkmark $\times$   $\times$   $\times$    0.004274   
                                      \checkmark \checkmark $\times$    0.326303   
Naive  $\times$ $\times$   $\times$   $\times$   \checkmark $\times$    0.307182   
                                                            \checkmark  0.284171   

                                                                             $R$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter                 
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.029667   
                                                 \checkmark $\times$    0.415946   
                           \checkmark $\times$   $\times$   $\times$    0.223773   
                                                 \checkmark $\times$    0.450644   
                                      \checkmark $\times$   $\times$    0.070961   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.447209   
                                      \checkmark \checkmark $\times$    0.453021   
                           \checkmark $\times$   $\times$   $\times$    0.001789   
                                      \checkmark \checkmark $\times$    0.448721   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.019983   
                                                 \checkmark $\times$    0.417106   
                                      \checkmark \checkmark $\times$    0.391645   
                           \checkmark $\times$   $\times$   $\times$    0.174418   
                                                 \checkmark $\times$    0.441260   
                                      \checkmark $\times$   $\times$    0.069524   
                                                 \checkmark $\times$    0.389665   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.442909   
                                      \checkmark \checkmark $\times$    0.449454   
                           \checkmark $\time

In [358]:
task_6_1_2_df.style.highlight_max(axis=0, props="textbf:--rwrap;").format(precision=2).to_latex(
    report_dir / "task_6_1_2.tex",
    # float_format="%.2f",
    caption=f"{mode} Set Result for Task 6.1.2 for various models and approaches.",
    label=f"tab:task:6_1_2{lbl_xtra}",
    clines="all;data",
    hrules=True
)
task_6_1_2_df

$P$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter                 
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.070592   
                                                 \checkmark $\times$    0.190624   
                           \checkmark $\times$   $\times$   $\times$    0.191776   
                                                 \checkmark $\times$    0.195908   
                                      \checkmark $\times$   $\times$    0.093587   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.217923   
                                      \checkmark \checkmark $\times$    0.215809   
                           \checkmark $\times$   $\times$   $\times$    0.000000   
                                      \checkmark \checkmark $\times$    0.235502   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.038288   
                                                 \checkmark $\times$    0.199215   
                                      \checkmark \checkmark $\times$    0.223402   
                           \checkmark $\times$   $\times$   $\times$    0.157536   
                                                 \checkmark $\times$    0.205233   
                                      \checkmark $\times$   $\times$    0.028776   
                                                 \checkmark $\times$    0.214220   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.210421   
                                      \checkmark \checkmark $\times$    0.231207   
                           \checkmark $\times$   $\times$   $\times$    0.000000   
                                      \checkmark \checkmark $\times$    0.225948   
Naive  $\times$ $\times$   $\times$   $\times$   \checkmark $\times$    0.217906   
                                                            \checkmark  0.252986   

                                                                             $R$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter                 
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.017458   
                                                 \checkmark $\times$    0.276970   
                           \checkmark $\times$   $\times$   $\times$    0.121088   
                                                 \checkmark $\times$    0.261460   
                                      \checkmark $\times$   $\times$    0.046834   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.311782   
                                      \checkmark \checkmark $\times$    0.307659   
                           \checkmark $\times$   $\times$   $\times$    0.000000   
                                      \checkmark \checkmark $\times$    0.315888   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.011593   
                                                 \checkmark $\times$    0.287676   
                                      \checkmark \checkmark $\times$    0.259287   
                           \checkmark $\times$   $\times$   $\times$    0.094749   
                                                 \checkmark $\times$    0.266178   
                                      \checkmark $\times$   $\times$    0.044443   
                                                 \checkmark $\times$    0.250897   
                \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                 \checkmark $\times$    0.299507   
                                      \checkmark \checkmark $\times$    0.302313   
                           \checkmark $\time

In [359]:
task_6_2_1_df.style.highlight_max(axis=0, props="textbf:--rwrap;").format(precision=2).to_latex(
    report_dir / "task_6_2_1.tex",
    # float_format="%.2f",
    caption=f"{mode} Set Result for Task 6.2.1 for various models and approaches.",
    label=f"tab:task:6_2_1{lbl_xtra}",
    clines="all;data",
    hrules=True
)
task_6_2_1_df

$P$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter               
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.000664   
                                                 \checkmark $\times$  0.000664   
                                      \checkmark $\times$   $\times$  0.111059   
                                                 \checkmark $\times$  0.111059   
                           \checkmark $\times$   $\times$   $\times$  0.093429   
                                                 \checkmark $\times$  0.093429   
                                      \checkmark $\times$   $\times$  0.037313   
                                                 \checkmark $\times$  0.037313   
                \checkmark $\times$   $\times$   $\times$   $\times$  0.000000   
                                                 \checkmark $\times$  0.000000   
                           \checkmark $\times$   $\times$   $\times$  0.000000   
                                      \checkmark \checkmark $\times$  0.000000   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.000035   
                                                 \checkmark $\times$  0.000035   
                                      \checkmark $\times$   $\times$  0.077010   
                                                 \checkmark $\times$  0.077010   
                           \checkmark $\times$   $\times$   $\times$  0.057597   
                                                 \checkmark $\times$  0.057597   
                                      \checkmark $\times$   $\times$  0.041115   
                                                 \checkmark $\times$  0.041115   
                \checkmark $\times$   $\times$   $\times$   $\times$  0.000000   
                                                 \checkmark $\times$  0.000000   
                                      \checkmark \checkmark $\times$  0.000000   
                           \checkmark $\times$   $\times$   $\times$  0.000000   
                                      \checkmark \checkmark $\times$  0.000000   

                                                                           $R$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter               
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.008862   
                                                 \checkmark $\times$  0.008862   
                                      \checkmark $\times$   $\times$  0.075825   
                                                 \checkmark $\times$  0.075825   
                           \checkmark $\times$   $\times$   $\times$  0.070922   
                                                 \checkmark $\times$  0.070922   
                                      \checkmark $\times$   $\times$  0.047481   
                                                 \checkmark $\times$  0.047481   
                \checkmark $\times$   $\times$   $\times$   $\times$  0.000000   
                                                 \checkmark $\times$  0.000000   
                           \checkmark $\times$   $\times$   $\times$  0.000000   
                                      \checkmark \checkmark $\times$  0.000000   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.001374   
                                                 \checkmark $\times$  0.001374   
                                      \checkmark $\times$   $\times$  0.061264   
                                                 \checkmark $\times$  0.061264   
                           \checkmark $\times$   $\times$   $\times$  0.054817   
                                                 \checkmark $\times$  0.054817   
                                      \checkmark $\times$   $\times$  0.045654   
                                                 \checkmark $\times$  0.045654   
                \checkmark $\times$   $\times$   $\times

In [360]:
task_6_2_2_df

$P$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter               
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.001294   
                                                 \checkmark $\times$  0.001294   
                                      \checkmark $\times$   $\times$  0.045429   
                                                 \checkmark $\times$  0.045429   
                           \checkmark $\times$   $\times$   $\times$  0.051441   
                                                 \checkmark $\times$  0.051441   
                                      \checkmark $\times$   $\times$  0.020579   
                                                 \checkmark $\times$  0.020579   
                \checkmark $\times$   $\times$   $\times$   $\times$  0.000212   
                                                 \checkmark $\times$  0.000228   
                           \checkmark $\times$   $\times$   $\times$  0.009127   
                                      \checkmark \checkmark $\times$  0.018646   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.000000   
                                                 \checkmark $\times$  0.000000   
                                      \checkmark $\times$   $\times$  0.028328   
                                                 \checkmark $\times$  0.028328   
                           \checkmark $\times$   $\times$   $\times$  0.032726   
                                                 \checkmark $\times$  0.032726   
                                      \checkmark $\times$   $\times$  0.017780   
                                                 \checkmark $\times$  0.017780   
                \checkmark $\times$   $\times$   $\times$   $\times$  0.000000   
                                                 \checkmark $\times$  0.000000   
                                      \checkmark \checkmark $\times$  0.012071   
                           \checkmark $\times$   $\times$   $\times$  0.020742   
                                      \checkmark \checkmark $\times$  0.000515   

                                                                           $R$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter               
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.022692   
                                                 \checkmark $\times$  0.022692   
                                      \checkmark $\times$   $\times$  0.034707   
                                                 \checkmark $\times$  0.034707   
                           \checkmark $\times$   $\times$   $\times$  0.037719   
                                                 \checkmark $\times$  0.037719   
                                      \checkmark $\times$   $\times$  0.032821   
                                                 \checkmark $\times$  0.032821   
                \checkmark $\times$   $\times$   $\times$   $\times$  0.005325   
                                                 \checkmark $\times$  0.007692   
                           \checkmark $\times$   $\times$   $\times$  0.007761   
                                      \checkmark \checkmark $\times$  0.003238   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.000000   
                                                 \checkmark $\times$  0.000000   
                                      \checkmark $\times$   $\times$  0.038632   
                                                 \checkmark $\times$  0.038632   
                           \checkmark $\times$   $\times$   $\times$  0.037040   
                                                 \checkmark $\times$  0.037040   
                                      \checkmark $\times$   $\times$  0.025772   
                                                 \checkmark $\times$  0.025772   
                \checkmark $\times$   $\times$   $\times

In [361]:
task_6_2_2_df.style.highlight_max(axis=0, props="textbf:--rwrap;").format(precision=2).to_latex(
    report_dir / "task_6_2_2.tex",
    # float_format="%.2f",
    caption=f"{mode} Set Result for Task 6.2.2 for various models and approaches.",
    label=f"tab:task:6_2_2{lbl_xtra}",
    clines="all;data",
    hrules=True
)
task_6_2_2_df

$P$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter               
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.001294   
                                                 \checkmark $\times$  0.001294   
                                      \checkmark $\times$   $\times$  0.045429   
                                                 \checkmark $\times$  0.045429   
                           \checkmark $\times$   $\times$   $\times$  0.051441   
                                                 \checkmark $\times$  0.051441   
                                      \checkmark $\times$   $\times$  0.020579   
                                                 \checkmark $\times$  0.020579   
                \checkmark $\times$   $\times$   $\times$   $\times$  0.000212   
                                                 \checkmark $\times$  0.000228   
                           \checkmark $\times$   $\times$   $\times$  0.009127   
                                      \checkmark \checkmark $\times$  0.018646   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.000000   
                                                 \checkmark $\times$  0.000000   
                                      \checkmark $\times$   $\times$  0.028328   
                                                 \checkmark $\times$  0.028328   
                           \checkmark $\times$   $\times$   $\times$  0.032726   
                                                 \checkmark $\times$  0.032726   
                                      \checkmark $\times$   $\times$  0.017780   
                                                 \checkmark $\times$  0.017780   
                \checkmark $\times$   $\times$   $\times$   $\times$  0.000000   
                                                 \checkmark $\times$  0.000000   
                                      \checkmark \checkmark $\times$  0.012071   
                           \checkmark $\times$   $\times$   $\times$  0.020742   
                                      \checkmark \checkmark $\times$  0.000515   

                                                                           $R$  \
Model  Beams    NE FT      RAG        LoRA       Naive      Filter               
3.1 8B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.022692   
                                                 \checkmark $\times$  0.022692   
                                      \checkmark $\times$   $\times$  0.034707   
                                                 \checkmark $\times$  0.034707   
                           \checkmark $\times$   $\times$   $\times$  0.037719   
                                                 \checkmark $\times$  0.037719   
                                      \checkmark $\times$   $\times$  0.032821   
                                                 \checkmark $\times$  0.032821   
                \checkmark $\times$   $\times$   $\times$   $\times$  0.005325   
                                                 \checkmark $\times$  0.007692   
                           \checkmark $\times$   $\times$   $\times$  0.007761   
                                      \checkmark \checkmark $\times$  0.003238   
3.2 3B $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.000000   
                                                 \checkmark $\times$  0.000000   
                                      \checkmark $\times$   $\times$  0.038632   
                                                 \checkmark $\times$  0.038632   
                           \checkmark $\times$   $\times$   $\times$  0.037040   
                                                 \checkmark $\times$  0.037040   
                                      \checkmark $\times$   $\times$  0.025772   
                                                 \checkmark $\times$  0.025772   
                \checkmark $\times$   $\times$   $\times

In [362]:
task_6_1_1_df.columns

Index(['$P$', '$R$', '$F_1$', '$P_{micro}$', '$R_{micro}$', '$F_{1,micro}$'], dtype='object')

In [363]:
import re

from constrerl.utils import calculate_improvements

pairs = {
    "Task 6.1": [task_6_1_1_df, task_6_1_2_df],
    "Task 6.2": [task_6_2_1_df, task_6_2_2_df],
}
for task_name, (df1, df2) in pairs.items():
    if df1.empty or df2.empty:
        print(f"Skipping {task_name} due to empty DataFrame.")
        continue
    task_name_formatted = re.sub(r"(\.| )", "_", task_name).strip().lower()
    disambiguation_changes = calculate_improvements(df1, df2)
    disambiguation_changes.style.format(
        lambda v: (
            f"+\\textcolor{{DarkGreen}}{{{v:.2f}}}"
            if v > 0
            else f"\\textcolor{{DarkRed}}{{{v:.2f}}}"
            if isinstance(v, float)
            else v
        )
    ).to_latex(
        report_dir / f"{task_name_formatted}_disambiguation.tex",
        caption=f"{mode} Set Disambiguation Changes for {task_name}",
        label=f"tab:task:{task_name_formatted}{lbl_xtra}_disambiguation",
        clines="all;data",
        hrules=True,
    )
disambiguation_changes

$F_1$  \
Model  Beams    NE FT    RAG        LoRA       Naive      Filter               
3.1 8B $\times$ $\times$ $\times$   \checkmark $\times$   $\times$ -0.039872   
                                               \checkmark $\times$ -0.039872   
                         \checkmark $\times$   $\times$   $\times$ -0.029455   
                                               \checkmark $\times$ -0.029455   
3.2 3B $\times$ $\times$ $\times$   \checkmark $\times$   $\times$ -0.022342   
                                               \checkmark $\times$ -0.022342   
                         \checkmark $\times$   $\times$   $\times$ -0.018422   
                                               \checkmark $\times$ -0.018422   
                                    \checkmark $\times$   $\times$ -0.011716   
                                               \checkmark $\times$ -0.011716   

                                                                    $F_{1,micro}$  \
Model  Beams    NE FT    RAG        LoRA       Naive      Filter                    
3.1 8B $\times$ $\times$ $\times$   \checkmark $\times$   $\times$      -0.049461   
                                               \checkmark $\times$      -0.049461   
                         \checkmark $\times$   $\times$   $\times$      -0.032104   
                                               \checkmark $\times$      -0.032104   
3.2 3B $\times$ $\times$ $\times$   \checkmark $\times$   $\times$      -0.049691   
                                               \checkmark $\times$      -0.049691   
                         \checkmark $\times$   $\times$   $\times$      -0.021077   
                                               \checkmark $\times$      -0.021077   
                                    \checkmark $\times$   $\times$      -0.052747   
                                               \checkmark $\times$      -0.052747   

                                                                    Relative $F_1$  \
Model  Beams    NE FT    RAG        LoRA       Naive      Filter                     
3.1 8B $\times$ $\times$ $\times$   \checkmark $\times$   $\times$       -0.614039   
                                               \checkmark $\times$       -0.614039   
                         \checkmark $\times$   $\times$   $\times$       -0.463968   
                                               \checkmark $\times$       -0.463968   
3.2 3B $\times$ $\times$ $\times$   \checkmark $\times$   $\times$       -0.456127   
                                               \checkmark $\times$       -0.456127   
                         \checkmark $\times$   $\times$   $\times$       -0.402148   
                                               \checkmark $\times$       -0.402148   
                                    \checkmark $\times$   $\times$       -0.518017   
                                               \checkmark $\times$       -0.518017   

                                                                    Relative $F_{1,micro}$  
Model  Beams    NE FT    RAG        LoRA       Naive      Filter                            
3.1 8B $\times$ $\times$ $\times$   \checkmark $\times$   $\times$               -0.434143  
                                               \checkmark $\times$               -0.434143  
                         \checkmark $\times$   $\times$   $\times$               -0.424432  
                                               \checkmark $\times$               -0.424432  
3.2 3B $\times$ $\times$ $\times$   \checkmark $\times$   $\times$               -0.436998  
                                               \checkmark $\times$               -0.436998  
                         \checkmark $\times$   $\times$   $\times$               -0.321978  
                                               \checkmark $\times$               -0.321978  
                                    \checkmark $\times$   $\times$               -0.509830  
                                 